# 06.03 · Chunking strategies

**Module 06 · Documents, Embeddings & RAG** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lesson 06.02. |
| 🔑 Keys | none (no model calls in this lesson) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** why we split documents, and what chunk size and overlap trade off.
2. **Use** the three main splitters: recursive (characters), Markdown headers, and tokens.
3. **Keep** metadata through splitting (`source`, `page`, `section`) and give each chunk a stable id.
4. **Tune** size and overlap by **measuring** retrieval on a golden set, not by guessing.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Text splitters](https://docs.langchain.com/oss/python/integrations/splitters) | Which splitter for which content; `RecursiveCharacterTextSplitter` is the recommended default |
| [Recursive text splitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) | Separators, `chunk_size`, `chunk_overlap`, `add_start_index` |
| [Markdown header splitter](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter) | Split on headings and keep them as metadata |
| [Split by tokens](https://docs.langchain.com/oss/python/integrations/splitters/split_by_token) | Counting size in model tokens (`from_tiktoken_encoder`) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langchain-text-splitters>=1.1.3,<2" "pypdf>=6.20.1,<7" "beautifulsoup4>=4.15.0,<5"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "langchain-text-splitters",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/06-rag/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out. Set these BEFORE importing chromadb/fastembed.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma: no usage telemetry.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: HF cache can't symlink without Developer Mode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")        # HF tokenizers' fork warning in Jupyter.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# tqdm (used by the model downloader) asks for ipywidgets to draw a progress bar; text output is fine.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
# Step 5: The course corpus: 24 Markdown files + 8 PDFs (23 pages) about fictional Nimbus weather stations.
import json

RAG_DIR = REPO_ROOT / "data" / "samples" / "rag"
CORPUS_DIR = RAG_DIR / "corpus"
GOLDEN = [json.loads(line) for line in (RAG_DIR / "golden-set.jsonl").read_text(encoding="utf-8").splitlines()]
print("corpus files:", sum(1 for p in CORPUS_DIR.rglob("*") if p.is_file()), "| golden questions:", len(GOLDEN))

In [ ]:
# Step 6: Load the corpus (06.02).
from mastery.rag import load_corpus

docs = load_corpus(CORPUS_DIR)
print(len(docs), "documents")

**You should now see** `corpus files: 32 | golden questions: 20` and `47 documents`.

---
## Part 1 · Why split at all?

- **Matching:** one embedding for a whole 20-page manual is a blurry average; a 1-paragraph chunk has a sharp meaning that matches a specific question.
- **Context budget:** you send the top-k chunks to the model. Five 800-character chunks ≈ 1,000 tokens; five whole manuals could be 50,000.
- **Citations:** a small chunk points to the exact place (file, page, section).

Too small, and a chunk loses its context ("It lasts 3 years." What does?). Too big, and matching gets blurry and prompts get expensive. **Overlap** repeats the end of one chunk at the start of the next, so a sentence cut at a boundary still appears whole somewhere.

---
## Part 2 · `RecursiveCharacterTextSplitter`

It tries separators in order (`"\n\n"`, `"\n"`, `" "`, `""`): split by paragraphs; if a piece is still too long, by lines; then by words. So it keeps paragraphs together whenever it can.

In [ ]:
# 🧪 Offline: split one policy with two sizes; look at the boundaries.
from langchain_text_splitters import RecursiveCharacterTextSplitter

returns = next(d for d in docs if d.metadata["source"] == "policies/returns.md")
print("original:", len(returns.page_content), "chars")
for size, overlap in [(160, 0), (160, 50)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap, add_start_index=True)
    pieces = splitter.split_documents([returns])
    print(f"\nchunk_size={size} overlap={overlap} → {len(pieces)} chunks")
    for p in pieces:
        print(f"  start={p.metadata['start_index']:<4} {len(p.page_content):>3} chars | {p.page_content[:60]!r}")

**You should now see** four chunks for each setting, each ≤ 160 characters, with `start_index` (the character offset in the original). Paragraphs that fit stay whole; the last paragraph is too long and gets cut at a space. Look at the **last chunk**: without overlap it is the orphan `'shipping.'` (9 characters, useless on its own); with `overlap=50` it starts earlier (`start=414`) and reads `'Colombia; international customers pay return shipping.'`. Overlap only matters where a paragraph had to be cut, but there it saves the meaning.

`add_start_index=True` is cheap and useful: it lets you highlight the exact passage in the original file later.

---
## Part 3 · Markdown headers as structure (and metadata)

Our Markdown files have headings (`## Specifications`, `## Returns`). `MarkdownHeaderTextSplitter` splits **at headings** and stores them as metadata, so a chunk knows its section, which also helps retrieval ("Nimbus Pro > Specifications").

In [ ]:
# 🧪 Offline: split a product page on its headings.
from langchain_text_splitters import MarkdownHeaderTextSplitter

pro = next(d for d in docs if d.metadata["source"] == "products/nimbus-pro.md")
headers = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2")], strip_headers=False)
for section in headers.split_text(pro.page_content):
    print(section.metadata, "|", len(section.page_content), "chars |", section.page_content.splitlines()[0])

**You should now see** one section per heading, e.g. `{'h1': 'Nimbus Pro', 'h2': 'Specifications'} | ... | ## Specifications`. Note: `split_text` returns **new** `Document`s with only the header metadata; the file's `source` is gone. `mastery.rag.split_documents(preset="markdown")` copies the parent metadata back, adds `section="Nimbus Pro > Specifications"`, and then splits long sections recursively.

---
## Part 4 · Size in tokens, not characters

Models count **tokens**. English averages ~4 characters per token, but code, numbers and other languages differ. Token splitting makes chunk size match your context budget exactly. `tiktoken` (OpenAI's tokenizer) is a good proxy for most models; it isn't installed by default, so `mastery.rag` falls back to "4 characters ≈ 1 token".

In [ ]:
# 🧪 Offline (🌐 if tiktoken is installed: it downloads its 1.7 MB vocabulary once): three presets side by side.
from mastery.rag import chunk_stats, split_documents
from mastery.rag.splitters import _token_length

print("token counter:", _token_length()[1])
presets = {
    "recursive 800/100": split_documents(docs, "recursive", 800, 100),
    "markdown 800/100": split_documents(docs, "markdown", 800, 100),
    "tokens 200/25": split_documents(docs, "tokens", 200, 25),
}
for name, chunks in presets.items():
    print(f"{name:<18} {chunk_stats(chunks)}")
sample = next(c for c in presets["markdown 800/100"] if c.metadata.get("section"))
print("\nexample chunk metadata:", sample.metadata)

**You should now see** the token counter (`tiktoken cl100k_base` or `approx (4 chars/token)`), three rows of statistics. Our files are short (the longest is ~750 characters), so `recursive 800` and `tokens 200` keep every file or PDF page whole (47 chunks = 47 documents), while the Markdown preset splits at headings (73 chunks), and an example chunk whose metadata has `source`, `type`, `title`, `section`, `start_index`, `chunk` and `chunk_id`.

The **`chunk_id`** (`"products/nimbus-pro.md#p0-c2"` = file, page, chunk number) is stable: same files + same settings → same ids. 06.05 uses it to **upsert** instead of duplicating chunks when you re-run indexing.

---
## Part 5 · Tune by measuring

Which size is best? Don't guess: measure retrieval on the golden set (20 questions with the expected source file; 06.09 explains the metrics). **recall@k** = share of expected files found in the top k; **MRR** rewards putting the right file first.

In [ ]:
# 🧪 Offline: a small grid. HashingEmbeddings (word-based, instant) makes the grid free to run.
from mastery.memory import HashingEmbeddings
from mastery.rag import build_index, evaluate_retrieval, retrieve

embeddings = HashingEmbeddings(dims=1024)
rows = []
for preset, size, overlap in [("recursive", 300, 50), ("recursive", 800, 100), ("recursive", 150, 30),
                              ("markdown", 800, 100), ("tokens", 200, 25)]:
    chunks = split_documents(docs, preset, size, overlap)
    store = build_index(chunks, embeddings)
    r1 = evaluate_retrieval(lambda q: retrieve(store, q, k=1), GOLDEN, k=1)
    r5 = evaluate_retrieval(lambda q: retrieve(store, q, k=5), GOLDEN, k=5)
    rows.append((f"{preset} {size}/{overlap}", len(chunks), r1["recall"], r5["recall"], r5["mrr"]))

print(f"{'setting':<20}{'chunks':>7}{'recall@1':>10}{'recall@5':>10}{'MRR':>7}")
for row in rows:
    print(f"{row[0]:<20}{row[1]:>7}{row[2]:>10.2f}{row[3]:>10.2f}{row[4]:>7.2f}")

**You should now see** five rows. recall@5 is high everywhere (our questions share words with the answers, and the corpus is small); the differences show at **recall@1 and MRR**: is the right file **first**? On our run the spread is small (recall@1 between 0.72 and 0.78, MRR 0.88–0.91): this corpus is tiny and its files are short, so every setting finds the right file somewhere in the top 5. Medium chunks or whole short files did as well as anything. On long documents (real manuals) whole-file chunks are impossible, and the differences get much bigger: that's when this grid earns its keep.

### What just happened?

You turned a style question ("what chunk size?") into a number. With 16 in-corpus questions, one question is 6 points: treat small differences as noise. With real embeddings (06.04) the numbers change, so re-run the grid when you change the embedding model.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `ValueError: Got a larger chunk overlap than chunk size` | `chunk_overlap >= chunk_size` | Overlap ≈ 10–20% of the size |
| Chunks lost `source`/`page` | `MarkdownHeaderTextSplitter.split_text` returns new Documents | Copy the parent metadata (as `split_documents` does) |
| Chunks contain only `## Heading` | Tiny sections | Merge small sections, or use recursive splitting for that file |
| A table or code block cut in half | Character splitting ignores structure | Larger chunks for those files, or a structure-aware splitter |
| `tiktoken` hangs or fails behind a proxy | It downloads its vocabulary on first use | The SSL fix in 00.02 Step 4b, or use the approximate counter |
| Duplicate chunks after re-indexing | Random ids | Stable ids (`chunk_id`) + upsert (06.05) |

## ⚠️ Production notes

> ⚠️ **Production note: chunking is part of the index version.**
> Changing the splitter or its settings changes every chunk and id. Store the settings with the index (name it e.g. `nimbus-md800-bge-small`) and rebuild from the source documents.

> ⚠️ **Production note: measure on your own questions.**
> Defaults (≈ 500–1,000 characters, 10–20% overlap) are a starting point. A golden set of real user questions is the only reliable way to tune.

## 🏋️ Exercises

1. **(Easy)** Split `policies/warranty.md` with `chunk_size=200, chunk_overlap=0`. Is any sentence cut in the middle?
2. **(Medium)** Add `("recursive", 500, 0)` and `("recursive", 500, 100)` to the grid. Does overlap change anything here?
3. **(Medium)** Which chunk of the Markdown preset is the shortest? Print its `chunk_id` and text. Is it useful on its own?
4. **(Stretch)** Write a `merge_small(chunks, min_chars=150)` that appends a too-short chunk to the next chunk of the same `source`. Re-run the grid with it.

<details>
<summary><b>Show solutions</b></summary>

**1.** `RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0).split_documents([doc])`: paragraphs that fit stay whole; a longer paragraph is split at a space, possibly mid-sentence. Overlap reduces the damage.

**2.** Usually little on this corpus (short files). Overlap matters more with long documents where answers straddle boundaries.

**3.** `min(presets["markdown 800/100"], key=lambda c: len(c.page_content))`: often a heading with one line. It retrieves poorly and wastes a slot in the top-k.

**4.** Loop over the chunks, keep a buffer per `source`, and concatenate while `len(buffer) < min_chars`; keep the first chunk's metadata and id. Expect slightly better MRR and fewer chunks.
</details>

## ✅ Checkpoint

- [ ] I can explain what chunk size and overlap trade off.
- [ ] I used the recursive, Markdown-header and token splitters, and kept metadata.
- [ ] Every chunk has a stable `chunk_id`.
- [ ] I chose settings by measuring recall@k / MRR on the golden set.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Which splitter does the [text splitters page](https://docs.langchain.com/oss/python/integrations/splitters) recommend as the default today, and which structure-aware splitters are listed (Markdown, HTML, code, JSON)? Log the answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Split for sharper matching, smaller prompts and precise citations.
- Recursive splitting is the default; split Markdown on headings; count tokens when budgets matter.
- Keep metadata and stable ids; tune with a golden set, not by feel.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 06.04 · Embeddings: local vs hosted** → [`06-04-embeddings.ipynb`](06-04-embeddings.ipynb).

[Module 06 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)